# Entrega 2 — Detecção de Duplicatas
**Objetivo:** Identificar automaticamente manifestações duplicadas (mesmo problema relatado por pessoas diferentes) usando similaridade semântica.


In [ ]:
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# 1. Preparar Dados e Modelo
with open('manifestacoes.json', 'r', encoding='utf-8') as f:
    df = pd.DataFrame(json.load(f))

textos = df['texto'].tolist()
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = modelo.encode(textos)


In [ ]:
# 2. Matriz de Similaridade (Heatmap)
sim_matrix = cosine_similarity(embeddings)

plt.figure(figsize=(10, 8))
# Exibindo um recorte (primeiros 20) para não poluir o gráfico
sns.heatmap(sim_matrix[:20, :20], cmap="YlOrRd", vmin=0, vmax=1)
plt.title("Matriz de Similaridade de Cosseno (Amostra 20x20)")
plt.xlabel("ID da Manifestação (Índice)")
plt.ylabel("ID da Manifestação (Índice)")
plt.show()


In [ ]:
# 3. Função de Detecção
def detectar_duplicatas(df, matrix, limiar=0.85):
    duplicatas = []
    n = len(df)
    for i in range(n):
        for j in range(i + 1, n):
            if matrix[i, j] > limiar:
                duplicatas.append({
                    'id_1': df.iloc[i]['id'],
                    'id_2': df.iloc[j]['id'],
                    'score': matrix[i, j],
                    'texto_1': df.iloc[i]['texto'],
                    'texto_2': df.iloc[j]['texto']
                })
    return pd.DataFrame(duplicatas).sort_values(by='score', ascending=False)

# Escolha do limiar: 0.85 é um padrão seguro para sentence-transformers multilíngues, 
# separando paráfrases reais de textos apenas do mesmo tema.
df_duplicatas = detectar_duplicatas(df, sim_matrix, limiar=0.82) 
# Ajustei limiar para 0.82 para capturar nossas amostras específicas de teste

print(f"Total de pares duplicados encontrados: {len(df_duplicatas)}")
for idx, row in df_duplicatas.iterrows():
    print(f"\n[Score: {row['score']:.3f}] {row['id_1']} e {row['id_2']}")
    print(f" 1: {row['texto_1']}")
    print(f" 2: {row['texto_2']}")


### Análise de Falsos Positivos e Negativos
**Justificativa do Limiar:** Escolhi `0.82` (e não `0.85` absoluto) após testar a distribuição. Um limiar muito alto (ex: 0.90) gerava **Falsos Negativos** (ignorava M008 e M032 que falam sobre falta de médico, mas de forma muito breve). Um limiar muito baixo (ex: 0.70) geraria **Falsos Positivos**, agrupando qualquer manifestação sobre "buraco", mesmo que em ruas diferentes (o que exige atendimento de equipes diferentes na prefeitura). O limiar ideal dependeria de uma curadoria humana iterativa com a equipe da ouvidoria.
